# 4 · Training: Split, Hyperparameter, Fitten

Pro Horizont und Fold:

1. **Hyperparameter wählen:** jeden Kandidaten (z.B. jedes $\lambda$) auf `tune_fit` fitten und auf `tune_eval` vergleichen
2. **Fitten:** mit dem besten Kandidaten auf `train` neu fitten
3. **Prognostizieren:** Gewichte für die Testzeilen → kombinierte Prognose (out-of-sample)

Gespeichert wird alles in `results/runs/<RUN>/`, ausgewertet in 05.

### Splits

| Split | Ablauf | wofür |
|---|---|---|
| **`walkforward`** (Standard) | jeden Monat neu trainieren, nächsten Monat prognostizieren. $\lambda$ wird auf den letzten `TUNE_DAYS` des Trainings gewählt. | echte Out-of-Sample-Prognosen über einen langen Zeitraum |
| `fixed` | train / val / test wie bisher. $\lambda$ auf val gewählt und auf val ausgewertet (leicht optimistisch). | Vergleich mit den alten Ergebnissen |

**Kein Leakage:** Eine Prognose für das Zieldatum $T$ hat den Ursprung $T-h$. Dort sind nur Ziele $\le T-h$ bekannt. Training und `tune_fit` enden deshalb mindestens $h$ Tage vor der ersten ausgewerteten Zeile (plus `GAP_DAYS`).

**Kontamination (echte Daten):** Liegt das Training vor dem Release der Foundation Models, können diese dort zu gut aussehen. `TRAIN_START` beschränkt das Training auf spätere Zieldaten, `WINDOW_DAYS` macht das Fenster rollierend statt expandierend.

### Kombinationsmethoden

Alle mischen Mittelwert **und** Quantile mit denselben Gewichten $w \ge 0$, $\sum w = 1$.

| Methode | Gewichte | Parameter |
|---|---|---|
| Gleichgewicht | $1/M$ | 0 |
| Auswahl | One-Hot auf den besten Experten im Training | 0 |
| Inverse Fehler | $w_m \propto 1/\overline{\text{QLIKE}}_m$ im Training (Bates/Granger) | 0 |
| Inverse Fehler 90T | wie oben, nur die letzten 90 Tage | 0 |
| Inverse jüngste Fehler | $w_{m,t} \propto 1/L_{m,t}$ (EWMA-QLIKE), jeden Tag neu, ohne Training | 0 |
| Statisch | $\mathrm{softmax}(b)$, auf den Loss trainiert | M |
| Statisch 365T | wie oben, nur das letzte Jahr (rollierend) | M |
| **Gating linear** | $\mathrm{softmax}(b + W x_t)$ | $M + Mk$ |
| Gating linear [Gruppe] | nur eine Feature-Gruppe (Ablation: woher kommt ein Gewinn?) | |
| Gating MLP | $\mathrm{softmax}(b + f(x_t))$, eine versteckte Schicht, Mittel über mehrere Seeds | |

Die Baselines ohne Parameter müssen geschlagen werden, damit sich ein Gate lohnt.

**Der Code (Panel, Loss, Baselines, Gate, Folds) steht in `src/pipeline/combine.py`.** Die App trainiert damit jeden Monat «Statisch» und «Gating linear» neu, mit der Konfiguration aus `results/runs/binance_walkforward/config.json`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET = "binance"        # Name aus 01_daten.ipynb, z.B. "binance" oder "syn_regime_signal_s0"

DATA_DIR, RESULTS_DIR = Path("data"), Path("results")
INFO = json.loads((DATA_DIR / f"{DATASET}.json").read_text())
DAILY_DIR, FORECAST_DIR = Path(INFO["daily_dir"]), Path(INFO["forecast_dir"])
SIDS = INFO["sids"]
SYNTHETIC = INFO["info"]["source"] == "synthetic"
print(f"Datensatz {DATASET}: {len(SIDS)} Serien {SIDS}")

# Gemeinsamer Code mit der App: src/pipeline/ (Einstellungen für beide in pipeline/common.py)
from pipeline.common import QUANTILES, QCOLS, LV as _LV, HORIZONS
from pipeline import combine as cb

import time
from dataclasses import dataclass, field
from functools import partial
import itertools

import torch

SPLIT = "walkforward"       # "walkforward" oder "fixed"
RUN = f"{DATASET}_{SPLIT}"  # Name des Ordners in results/runs/
ALLOW_TEST = False          # Testperiode (ab test_start in data/<DATASET>.json) erst am Schluss öffnen!

# ---- Walk-Forward ----
EVAL_START = "2025-11-01" if not SYNTHETIC else None   # None = nach 60 % der Daten
EVAL_END = "2026-02-28" if not SYNTHETIC else None     # None = bis zum Ende
FREQ = "MS" if not SYNTHETIC else "QS"                 # Neutraining: "MS" monatlich, "QS" quartalsweise
TRAIN_START = None          # z.B. "2025-11-01": nur Zieldaten nach dem Release der Foundation Models
WINDOW_DAYS = None          # None = expandierendes Fenster
TUNE_DAYS = 90              # letzte Tage des Trainings für die Wahl von lambda
GAP_DAYS = 0                # zusätzliche Lücke (Mindestlücke ist h)

# ---- Fixed ----
TRAIN_END, VAL_END = "2025-10-31", "2026-02-28"

# ---- Modelle ----
ALPHA = 1.0                 # Gewicht des Pinball-Terms im Loss
LAMBDAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
MLP = True
MLP_HIDDEN, MLP_SEEDS = 16, 3
ABLATIONS = [["recent"], ["market"], ["exo"]]   # lineares Gate mit nur diesen Gruppen
LEGACY = False              # True: Methoden genau wie in 01_kombination (mit SPLIT = "fixed" und LEGACY in 03)

RUN_DIR = RESULTS_DIR / "runs" / RUN
cb.ALPHA = ALPHA             # Loss in pipeline/combine.py

Datensatz binance: 3 Serien ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']


## Panels laden

In [2]:
PANEL_DIR = RESULTS_DIR / "panels" / DATASET
PANEL_INFO = json.loads((PANEL_DIR / "panel.json").read_text())
POOL, GROUP_MAP = PANEL_INFO["pool"], PANEL_INFO["groups"]

Panel = cb.Panel        # Code: pipeline/combine.py
panels = {h: Panel(pd.read_parquet(PANEL_DIR / f"h{h}.parquet"), h, POOL, GROUP_MAP) for h in PANEL_INFO["horizons"]}
print("Pool:", POOL)
for h, P in panels.items():
    print(f"h={h}: {len(P):,} Zeilen, {len(P.features())} Features")

Pool: ['Chronos-2-full', 'TimesFM-2.5', 'HAR', 'GARCH']
h=1: 5,546 Zeilen, 16 Features
h=5: 5,546 Zeilen, 16 Features


## Loss

$$\mathcal{L} = \overline{\text{QLIKE}(y, \hat\mu)} + \alpha \cdot \overline{\text{Pinball}(y, \hat q) / s_t} \;(+\; \lambda \textstyle\sum W^2 \text{ beim Gate})$$

QLIKE bewertet den kombinierten Mittelwert, Pinball die kombinierten Quantile. $s_t$ ist das RV-Niveau der letzten 22 Tage, damit volatile Phasen und Serien das Training nicht dominieren.

In [3]:
from pipeline.common import qlike
from pipeline.combine import comb_loss_torch, comb_loss, last_days, T     # Code: pipeline/combine.py

## Baselines

Schnittstelle wie scikit-learn: `fit(panel_train)` lernt, `weights(panel)` gibt Gewichte `(n, M)` für beliebige Zeilen.

In [4]:
from pipeline.combine import Combiner, Equal, BestSingle, InverseLoss, RecentInverseLoss

## Gate

$w = \mathrm{softmax}(b + f(x_t))$, mit $f$ linear (`hidden=0`) oder als MLP mit tanh.

- **Start bei Gleichgewicht:** $b = 0$, letzte Schicht $= 0$.
- **Standardisierung** der Features nur mit Mittelwert und Streuung der Trainingszeilen des Folds. Fehlende Werte (Aufwärmphase) → 0.
- **L2-Strafe $\lambda$ nur auf die Gewichtsmatrizen** von $f$, nicht auf $b$ und nicht auf Biases. Grosses $\lambda$ → statische Gewichte. (Das alte Notebook hat beim MLP auch den Bias der ersten Schicht bestraft; `penalize_bias=True` stellt das wieder her.)
- **MLP:** mehrere Seeds, die Gewichte werden gemittelt. Ein einzelner Seed hängt stark von der zufälligen ersten Schicht ab.
- **`groups=()`** heisst keine Features, also statische Gewichte $\mathrm{softmax}(b)$.
- Optimiert wird full-batch mit L-BFGS.

In [5]:
from pipeline.combine import GateNet, Gate

## Methoden und Hyperparameter

In [6]:
from pipeline.combine import Method       # Name, make(**params) -> Combiner, grid der Hyperparameter


if LEGACY:
    METHODS = [
        Method("Gleichgewicht", Equal),
        Method("Auswahl", BestSingle),
        Method("Statisch", partial(Gate, groups=())),
        Method("Gating linear", Gate, {"lam": LAMBDAS}),
        Method("Gating MLP", partial(Gate, hidden=16, penalize_bias=True), {"lam": LAMBDAS}),
    ]
else:
    METHODS = [
        Method("Gleichgewicht", Equal),
        Method("Auswahl", BestSingle),
        Method("Inverse Fehler", InverseLoss),
        Method("Inverse Fehler 90T", partial(InverseLoss, window_days=90)),
        Method("Inverse jüngste Fehler", RecentInverseLoss),
        Method("Statisch", partial(Gate, groups=())),
        Method("Statisch 365T", partial(Gate, groups=(), window_days=365)),
        Method("Gating linear", Gate, {"lam": LAMBDAS}),
    ]
    for g in ABLATIONS:
        if all(x in GROUP_MAP for x in g):
            METHODS.append(Method(f"Gating linear [{'+'.join(g)}]", partial(Gate, groups=tuple(g)), {"lam": LAMBDAS}))
    if MLP:
        METHODS.append(Method("Gating MLP", partial(Gate, hidden=MLP_HIDDEN, seeds=MLP_SEEDS), {"lam": LAMBDAS}))

print([m.name for m in METHODS])

['Gleichgewicht', 'Auswahl', 'Inverse Fehler', 'Inverse Fehler 90T', 'Inverse jüngste Fehler', 'Statisch', 'Statisch 365T', 'Gating linear', 'Gating linear [recent]', 'Gating linear [market]', 'Gating MLP']


## Splits

In [7]:
from pipeline.combine import Fold


def make_folds(P):
    if SPLIT == "walkforward":
        folds = cb.walkforward_folds(P, EVAL_START, EVAL_END, FREQ, TUNE_DAYS, GAP_DAYS, TRAIN_START, WINDOW_DAYS)
    else:
        folds = cb.fixed_folds(P, TRAIN_END, VAL_END)
    test_start = INFO["test_start"]
    if test_start and not ALLOW_TEST:
        for f in folds:
            if (P.td[f.test] >= pd.Timestamp(test_start)).any():
                raise RuntimeError(f"Fold {f.label} wertet Zieldaten ab {test_start} aus. Die Testperiode ist "
                                   "gesperrt: EVAL_END früher setzen oder erst final ALLOW_TEST = True.")
    return folds


for h, P in panels.items():
    folds = make_folds(P)
    f0, f1 = folds[0], folds[-1]
    print(f"h={h}: {len(folds)} Folds, Test {P.td[f0.test].min().date()} bis {P.td[f1.test].max().date()}, "
          f"Training im ersten Fold: {f0.train.sum():,} Zeilen bis {P.td[f0.train].max().date()}")

h=1: 4 Folds, Test 2025-11-01 bis 2026-02-28, Training im ersten Fold: 4,646 Zeilen bis 2025-10-31
h=5: 4 Folds, Test 2025-11-01 bis 2026-02-28, Training im ersten Fold: 4,622 Zeilen bis 2025-10-27


## Hyperparameter wählen, fitten, prognostizieren

In [8]:
from pipeline.combine import fit_predict    # lambda auf tune_fit/tune_eval wählen, dann auf train fitten


def to_long(P, w, name, fold):
    df = P.meta[["sid", "h", "origin", "target_date", "y"]
                + [c for c in ["true_var", "regime"] if c in P.meta]].copy()
    df["model"], df["fold"] = name, fold
    df["mean"] = (w * P.mean).sum(1)
    df[QCOLS] = np.einsum("nm,nmq->nq", w, P.Q)
    return df


out, wts, params, tuning, coefs = [], [], [], [], []
t0 = time.time()
for h, P in panels.items():
    folds = make_folds(P)
    for i, fold in enumerate(folds):
        Pt = P.take(fold.test)
        M = len(POOL)
        for j, m in enumerate(POOL):                          # Experten als One-Hot (gleiche Zeilen)
            out.append(to_long(Pt, np.eye(M)[np.full(len(Pt), j)], m, fold.label))
        for method in METHODS:
            c, best, tun = fit_predict(P, method, fold)
            w = c.weights(Pt)
            out.append(to_long(Pt, w, method.name, fold.label))
            wdf = Pt.meta[["sid", "h", "origin", "target_date"] + (["regime"] if "regime" in Pt.meta else [])].copy()
            wdf["model"], wdf["fold"] = method.name, fold.label
            wdf[POOL] = w
            wts.append(wdf)
            params.append({"h": h, "fold": fold.label, "Methode": method.name, **best})
            tuning += tun
            if method.name == "Gating linear":
                coefs.append(c.coef().rename_axis("feature").reset_index().assign(h=h, fold=fold.label))
        print(f"\rh={h}: Fold {i + 1}/{len(folds)} ({time.time() - t0:.0f}s)", end="")
    print()

RUN_DIR.mkdir(parents=True, exist_ok=True)
pd.concat(out, ignore_index=True).to_parquet(RUN_DIR / "forecasts.parquet")
pd.concat(wts, ignore_index=True).to_parquet(RUN_DIR / "weights.parquet")
params = pd.DataFrame(params)
params.to_csv(RUN_DIR / "params.csv", index=False)
tuning = pd.DataFrame(tuning)
tuning.to_csv(RUN_DIR / "tuning.csv", index=False)
if coefs:
    pd.concat(coefs, ignore_index=True).to_parquet(RUN_DIR / "coef.parquet")
config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (str, int, float, bool, list, type(None)))}
(RUN_DIR / "config.json").write_text(json.dumps({**config, "GROUP_MAP": GROUP_MAP, "panel": PANEL_INFO["config"]},
                                                indent=2, default=str))
print("gespeichert in", RUN_DIR)

/var/folders/fc/r822p1_s5v5_mswzrz420sj00000gn/T/ipykernel_48300/2174998127.py:17: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:214.)
  return torch.as_tensor(np.asarray(a, dtype=float))


h=1: Fold 1/4 (20s)

h=1: Fold 2/4 (38s)

h=1: Fold 3/4 (58s)

h=1: Fold 4/4 (76s)


h=5: Fold 1/4 (96s)

h=5: Fold 2/4 (111s)

h=5: Fold 3/4 (125s)

h=5: Fold 4/4 (140s)
gespeichert in results/runs/binance_walkforward


## Welches λ wurde gewählt?

Wird oft das grösste λ gewählt, ist das Gate faktisch statisch. Die Features senken dann den Fehler auf neuen Daten nicht.

In [9]:
lam_methods = [m.name for m in METHODS if "lam" in m.grid]
if lam_methods:
    display(params[params["Methode"].isin(lam_methods)]
            .pivot_table(index="Methode", columns=["h", "lam"], values="fold", aggfunc="count", fill_value=0))
if len(tuning):
    display(tuning.pivot_table(index=["h", "lam"], columns="Methode", values="Loss tune", aggfunc="mean").round(4))

h                           1                    5                     
lam                    0.0001 0.0010 0.0100 0.0001 0.0010 0.0100 1.0000
Methode                                                                
Gating MLP                  0      4      0      1      1      2      0
Gating linear               0      3      1      0      2      2      0
Gating linear [market]      4      0      0      0      3      1      0
Gating linear [recent]      2      2      0      0      0      3      1

Methode   Gating MLP  Gating linear  Gating linear [market]  \
h lam                                                         
1 0.0001      0.5137         0.5177                  0.5102   
  0.0010      0.5053         0.5106                  0.5124   
  0.0100      0.5278         0.5150                  0.5201   
  0.1000      0.5278         0.5252                  0.5268   
  1.0000      0.5278         0.5276                  0.5277   
5 0.0001      0.6138         0.6305                  0.6121   
  0.0010      0.6147         0.6105                  0.6004   
  0.0100      0.6112         0.6111                  0.6039   
  0.1000      0.6210         0.6178                  0.6161   
  1.0000      0.6210         0.6201                  0.6205   

Methode   Gating linear [recent]  
h lam                             
1 0.0001                  0.5220  
  0.0010                  0.5211  
  0.0100                  0.5237  
  0.1000                  0.5271  
  1.0000                  0.5277  
5 0.0001                  0.6275  
  0.0010                  0.6210  
  0.0100                  0.6095  
  0.1000                  0.6162  
  1.0000                  0.6204